In [ ]:
# Databricks notebook source
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import StructType, StringType, DoubleType

# Schema matching both Elecz and ENTSO-E
schema = StructType() \
    .add("timestamp", StringType()) \
    .add("zone", StringType()) \
    .add("price", DoubleType()) \
    .add("source", StringType())
    
    # 1. Retrieve the password from the Secret Scope
aiven_password = dbutils.secrets.get(scope="kafka-scope", key="aiven-password")

# 2. Dynamically construct the JAAS configuration string
jaas_config = f'kafkashaded.org.apache.kafka.common.security.scram.ScramLoginModule required username="avnadmin" password="{aiven_password}";'

# Read from Kafka
kafka_df = (spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka-26289383-md-f1a0.l.aivencloud.com:26452")
    .option("subscribe", "energy-prices")
    .option("startingOffsets", "latest")
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism", "SCRAM-SHA-256")
    .option("kafka.sasl.jaas.config", jaas_config)  # Use the dynamically constructed variable
    # Remove the kafka.ssl.ca.pem line to prevent the Databricks Free Edition error
        # Key: Point to the .pem file in the Volume
    .option("kafka.ssl.truststore.location", "/Volumes/workspace/default/energy_realtime/aiven_ca.pem")
    .option("kafka.ssl.truststore.type", "PEM")
    .load()
)

# Parse JSON
parsed_df = kafka_df.select(
    from_json(col("value").cast("string"), schema).alias("data")
).select("data.*")

# Write to Delta Table
# Replace the checkpoint location with the Volume path you created in Step 1
query = (parsed_df.writeStream
    .format("delta")
    .outputMode("append")
    .trigger(availableNow=True)  # <--- Key: use AvailableNow trigger
    .option("checkpointLocation", "/Volumes/workspace/default/energy_realtime/checkpoints")
    .toTable("energy_prices_realtime"))

query.awaitTermination()